Purpose :
- S3 Landing JSON -> Bronze Delta table
- Read from : s3://your-lakehouse-bucket/lakehouse/landing/douyin/api_raw/json/
- Write to : s3://your-lakehouse-bucket/lakehouse/bronze/douyin/aweme_delta/

In [0]:
from pyspark.sql import functions as F

bucket = "de-e2e-413612133697-ap-southeast-1-an"

landing_root = f"s3://{bucket}/lakehouse/landing/douyin/api_raw/json/"
bronze_path = f"s3://{bucket}/lakehouse/bronze/douyin/api_raw_delta/"

raw_text_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .text(landing_root)
    .select(
        F.col("_metadata.file_path").alias("source_file"),
        F.col("value").alias("raw_json_string"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
)

parsed_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(landing_root)
    .select(
        F.col("_metadata.file_path").alias("source_file"),
        F.col("pipeline"),
        F.col("source"),
        F.col("zone"),
        F.col("niche"),
        F.col("account_id"),
        F.col("account_type"),
        F.col("generated_at"),
        F.col("crawl_config"),
        F.col("media_manifest"),
        F.struct("*").alias("raw_struct")
    )
)

bronze_df = raw_text_df.join(parsed_df, on="source_file", how="inner")

display(bronze_df)

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .save(bronze_path)
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS de_e2e.bronze.douyin_api_raw
USING DELTA
LOCATION '{bronze_path}'
""")

In [0]:
%sql
SELECT source_file, account_id, generated_at
FROM de_e2e.bronze.douyin_api_raw
LIMIT 20;